In [1]:
#data and visuals
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import datetime


#modelling
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Imputer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

pd.options.display.max_rows = 4000

C:\Users\zmerritt\AppData\Local\Continuum\anaconda3\lib\site-packages\h5py\__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
#user defined lists
segments = ["Cheap + Comfy","Cheap + Curious","Curious","Comfy","Craver","Caviar"]
segments2 = ["Assigned Segment." + i for i in segments]

bain_10questions = [
    'I always choose higher end hotels',
    'On vacation I pride myself on discovering new places and things my friends have not seen before',
    'I choose more expensive hotels and accommodations to make sure I get the best service and amenities',
    'It is important for me to be in a luxurious setting and to be pampered on vacation',
    'I like to immerse myself in the local culture when on vacation',
    'The difference between luxury and standard hotels just is not big enough to justify the cost',
    'I always pay more for a higher class of room on my vacations',
    'I would rather spend as little as possible on accommodations so I have more money to spend on other parts of my vacation',
    'I think spending more for unique experiences is part of what makes a vacation special',
    'I like to go on vacations that are off the beaten path'
]

In [3]:
#user defined functions

#Function: multi_pred_enricher
def multi_pred_enricher(preds):
    """
    Purpose: transform multi-output prediction array into dataframe with predicted Bain segment
    """
    preds = pd.DataFrame(preds)
    preds.columns = segments2
    preds = preds[segments2]
    preds['Assigned Segment'] = ""
    preds['Assigned Segment (Model Value)'] = 0
    for i in segments2:
        #Assign segment if resulting model value is the maximum resulting value
        preds.loc[:,'Assigned Segment'] = np.where(
            preds['Assigned Segment (Model Value)'] >= preds[i], preds['Assigned Segment'], i)
        preds.loc[:,'Assigned Segment (Model Value)'] = np.where(
            preds['Assigned Segment'] == i, preds[i], preds['Assigned Segment (Model Value)'])

    preds['Assigned Segment'] = preds['Assigned Segment'].map(lambda x: x.lstrip('Assigned Segment.'))   
    
    return preds

#Function: multi_pred_score_visualizer
def multi_pred_score_visualizer(actual, preds):
    score = np.mean(actual == preds)
    cm = metrics.confusion_matrix(actual, preds)
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt=".0f", linewidths=.5, square = True, cmap = 'Blues_r'
               ,xticklabels = ["Caviar", "Cheap + Comfy","Cheap + Curious","Comfy","Craver","Curious"]
               ,yticklabels = ["Caviar", "Cheap + Comfy","Cheap + Curious","Comfy","Craver","Curious"]
               );
    plt.ylabel('Actual label');
    plt.xlabel('Predicted label');
    all_sample_title = 'Accuracy Score (test): {0}'.format(score.round(10))
    plt.suptitle(all_sample_title, size = 15);
    #plt.title('Accuracy Score (train): {0}'.format(train_score.round(10)), size = 11);
    plt.show()
    
    
def multi_feature_importance(model):
    # Plot feature importance
    feature_importance = model.feature_importances_
    # make importances relative to max importance
    feature_importance = 100.0 * (feature_importance / feature_importance.max())
    sorted_idx = np.argsort(feature_importance)
    pos = np.arange(sorted_idx.shape[0]) + .5
    plt.subplot(1, 2, 2)
    plt.barh(pos[-20:], feature_importance[sorted_idx][-20:], align='center')
    plt.yticks(pos[-20:], x_train_raw.columns[sorted_idx][-20:])
    plt.xlabel('Relative Importance')
    plt.title('Variable Importance')
    plt.show()
    

In [4]:
x_vars = pd.read_csv("../../data/bain_x_vars.csv")
y_vars = pd.read_csv("../../data/bain_y_vars.csv")

print("x_vars.shape:",x_vars.shape)
print("y_vars.shape:",y_vars.shape)

x_vars.shape: (616, 944)
y_vars.shape: (616, 90)


In [21]:
x_train, x_test, y_train, y_test = train_test_split(x_vars,
                                                    y_vars,
                                                    test_size=0.15, random_state=195)


In [22]:
x_train_raw = x_train
x_test_raw = x_test

train = pd.concat([x_train, y_train["Assigned Segment"]], axis = 1)
test = pd.concat([x_test, y_test["Assigned Segment"]], axis = 1)

#fill in missing data
imp = IterativeImputer(max_iter=10, random_state=0)
imp.fit(x_train)
x_train = imp.transform(x_train)
x_test = imp.transform(x_test)

#scale
sc = StandardScaler()
sc.fit(x_train)
x_train = sc.transform(x_train)
x_test = sc.transform(x_test)

sm = SMOTE(random_state=12, sampling_strategy = 'all', k_neighbors = 12)
x_train_res, y_train_res = sm.fit_sample(x_train, y_train["Assigned Segment"])
#x_train_resdf = pd.DataFrame(x_train_res)
#x_train_resdf.columns = x_train_raw.columns
#y_train_resdf = pd.DataFrame(y_train_res)
#y_train_resdf.columns = ["Assigned Segment"]

print("Shape - x_train:",x_train_resdf.shape)
print("Shape - x_test:",x_test.shape)
print("Shape - y_train:",y_train_resdf.shape)
print("Shape - y_test:",y_test.shape)

Shape - x_train: (1224, 944)
Shape - x_test: (93, 944)
Shape - y_train: (1224, 1)
Shape - y_test: (93, 90)


In [29]:
x_train_resdf = pd.DataFrame(x_train_res)
x_train_resdf.columns = x_train_raw.columns
y_train_resdf = pd.DataFrame(y_train_res)
y_train_resdf.columns = ["Assigned Segment"]

In [27]:
y_train_res

array(['Curious', 'Caviar', 'Comfy', ..., 'Curious', 'Curious', 'Curious'],
      dtype=object)

In [16]:
import scipy
x_train_sparse = scipy.sparse.csr_matrix(x_train_resdf)
x_test_sparse = scipy.sparse.csr_matrix(x_test)

In [ ]:
from tpot import TPOTClassifier
pipeline_optimizer = TPOTClassifier(generations=100, population_size=200, cv=5,
                                    random_state=42, verbosity=3, config_dict='TPOT sparse',
                                   n_jobs = -1)
pipeline_optimizer.fit(x_train_res, y_train_res)

13 operators have been imported by TPOT.


Skipped pipeline #32 due to time out. Continuing to the next pipeline.
Skipped pipeline #73 due to time out. Continuing to the next pipeline.
Skipped pipeline #112 due to time out. Continuing to the next pipeline.
Skipped pipeline #183 due to time out. Continuing to the next pipeline.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='logistic_regression' are not supported 

_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=2 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=3 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=0 Expected n_neighbors <= n_samples,  but n_samples = 50, n_neighbors = 51.
_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Craver'.
_pre_test decorator: _random_mutation_operator: num_test=2 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The

_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='logistic_regression' are not supported when dual=True, Parameters: penalty='l1', loss='logistic_regression', dual=True.
_pre_test decorator: _random_mutation_operator: num_test=1 Expected n_neighbors <= n_samples,  but n_samples = 50, n_neighbors = 100.
_pre_test decorator: _random_mutation_operator: num_test=0 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=1 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='hinge' is not supported, Parameters: penalty='l1', loss='hinge', dual=True.
_pre_test decorator: _random_mutation_operator: num_test=1 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is

_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='squared_hinge' are not supported when dual=True, Parameters: penalty='l1', loss='squared_hinge', dual=True.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=2 Unsupported set of arguments: The combination of penalty='l1' and loss='logistic_regression' are not supported when dual=True, Parameters: penalty='l1', loss='logistic_regression', dual=True.
_pre_test decorat

...

_pre_test decorator: _random_mutation_operator: num_test=4 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=5 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=1 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='hinge' is not supported, Parameters: penalty='l1', loss='hinge', dual=False.
_pre_test decorator: _random_mutation_operator: num_test=0 A sparse matrix was passed, but dense data is 

_pre_test decorator: _random_mutation_operator: num_test=6 Input X must be non-negative.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='logistic_regression' are not supported when dual=True, Parameters: penalty='l1', loss='logistic_regression', dual=True.
_pre_test decorator: _random_mutation_operator: num_test=1 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l2' and loss='hinge' are not supported when dual=False, Parameters: penalty='l2', loss='hinge', dual=False.
_pre_test decorator: _random_mutation_operator: num_test=0 Input X must be non-negative.
_pre_test decorator: _random_mu

_pre_test decorator: _random_mutation_operator: num_test=0 A sparse matrix was passed, but dense data is required. Use X.toarray() to convert to a dense numpy array..
_pre_test decorator: _random_mutation_operator: num_test=0 No feature in X meets the variance threshold 0.20000.
_pre_test decorator: _random_mutation_operator: num_test=0 index 51 is out of bounds for axis 0 with size 51.
_pre_test decorator: _random_mutation_operator: num_test=1 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='logistic_regression' are not supported when dual=True, Parameters: penalty='l1', loss='logistic_regression', dual=True.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=1 Found array with 0 feature(s) (shape=(50, 0)) while a m

_pre_test decorator: _random_mutation_operator: num_test=0 index 51 is out of bounds for axis 0 with size 51.
_pre_test decorator: _random_mutation_operator: num_test=0 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=1 A sparse matrix was passed, but dense data is required. Use X.toarray() to convert to a dense numpy array..
_pre_test decorator: _random_mutation_operator: num_test=2 Unsupported set of arguments: The combination of penalty='l1' and loss='squared_hinge' are not supported when dual=True, Parameters: penalty='l1', loss='squared_hinge', dual=True.
_pre_test decorator: _random_mutation_operator: num_test=0 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l2' and loss='hinge' are not supported when dual=False, Parameters: penalty='l2', loss='hinge', dual=False.
_pre_test deco

_pre_test decorator: _random_mutation_operator: num_test=0 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=1 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=2 could not convert string to float: 'Curious'.
_pre_test decorator: _random_mutation_operator: num_test=3 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=4 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=5 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_test=6 Found array with 0 feature(s) (shape=(50, 0)) while a minimum of 1 is required..
_pre_test decorator: _random_mutation_operator: num_tes

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectFromModel

steps = [
    ('imputer', IterativeImputer(max_iter=10, random_state=195)),
    ('scaler', StandardScaler()),
    ('reducer', PCA(n_components = .5, svd_solver = 'full', iterated_power = 100000, whiten = True, random_state=195)),
    ('selector', SelectFromModel(LinearSVC(C=.5, penalty="l1", dual=False))),
    ('model', RandomForestClassifier(n_estimators=200,
                                     max_depth=30,
                                     min_samples_leaf = 1,
                                     random_state=195))
]

model = Pipeline(steps)

model.fit(x_train_resdf, y_train_resdf['Assigned Segment'])
predictions = model.predict(x_test)
multi_pred_score_visualizer(y_test['Assigned Segment'],predictions)

In [18]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()

In [19]:
digits.data

array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ..., 10.,  0.,  0.],
       [ 0.,  0.,  0., ..., 16.,  9.,  0.],
       ...,
       [ 0.,  0.,  1., ...,  6.,  0.,  0.],
       [ 0.,  0.,  2., ..., 12.,  0.,  0.],
       [ 0.,  0., 10., ..., 12.,  1.,  0.]])